# 02 · Your first Iceberg table

**Goal:** create an Iceberg table, then *watch* what Iceberg writes to storage at each step — so table formats stop being magic. Then use the features plain Parquet couldn't give you in notebook 01: hidden partitioning, snapshots, time travel, `DELETE`, rollback and safe renames.

**Databricks link:** this is exactly what a Delta or Iceberg table on Databricks does under the hood. Databricks' *UniForm* and *managed Iceberg tables* use this same format.

**How to use:** run cells in order with **Shift + Enter**, read **What to notice** after each result. About 45 minutes. Keep the RustFS console open in another tab to browse the files as they appear: http://localhost:9001/rustfs/console/ (login `lakehouse` / `lakehouse-secret-key`, bucket `warehouse`).

## How an Iceberg table is built

```
 catalog (iceberg-rest)          "learn.yellow_trips → s3://warehouse/learn/yellow_trips/metadata/00002-….metadata.json"
     │  points to the CURRENT metadata file — that's all it stores
     ▼
 metadata.json                   schema, partition spec, list of ALL snapshots, current snapshot id
     │  one new file per commit
     ▼
 manifest list  (snap-….avro)    one per snapshot: which manifests make up this version
     │
     ▼
 manifests  (….-m0.avro)         list of data files + each file's partition, row count, min/max per column
     │
     ▼
 data files  (….parquet)         the actual rows — plain Parquet, like notebook 01
```

A **commit** = write new data files → write new manifests/manifest list → write a new metadata.json → ask the catalog to swap its pointer from the old metadata.json to the new one. That single pointer swap is atomic, which is why readers never see half a write.

> **Coming from Hive/Impala — the big shift:** in Hive, the **metastore** holds the table's location *and every partition's folder*, and the engine finds files by **listing folders** on HDFS. That's why you run `MSCK REPAIR TABLE` / `REFRESH` when files appear, and why tables with many partitions get slow. In Iceberg, the catalog holds **one pointer** per table, and Iceberg's own metadata files list **every data file** explicitly. No folder listing, no repair, no refresh: the table is whatever the current metadata.json says.

In [ ]:
from lab_helpers import get_spark, s3_tree, s3_json, s3_ls, scan_report
from pyspark.sql import functions as F
import pandas as pd

spark = get_spark("02-iceberg-first-table")
TABLE = "learn.yellow_trips"
PREFIX = "learn/yellow_trips/"

jan = spark.read.parquet("/home/lab/work/data/raw/yellow_tripdata_2024-01.parquet")
feb = spark.read.parquet("/home/lab/work/data/raw/yellow_tripdata_2024-02.parquet")

# Start clean so this notebook can be re-run from the top.
spark.sql(f"DROP TABLE IF EXISTS {TABLE} PURGE")
spark.sql("CREATE NAMESPACE IF NOT EXISTS learn")
spark.sql("SHOW NAMESPACES").show()

**What to notice:** a **namespace** is like a schema/database (`bronze`, `silver`, `gold` later). It lives in the catalog; there's no folder for it until a table writes files.

> **Coming from Hive/Impala:** a namespace is a Hive **database**. `CREATE NAMESPACE learn` = `CREATE DATABASE learn`; `learn.yellow_trips` reads exactly like `db.table`.

## 1. Create an empty table with hidden partitioning

`PARTITIONED BY (days(tpep_pickup_datetime))` tells Iceberg to group files by the **day** of the pickup time. Unlike notebook 01, there's **no extra `pickup_date` column** — Iceberg derives the day itself. That's *hidden partitioning*.

> **Coming from Hive/Impala:** compare with Hive's `PARTITIONED BY (dt STRING)`, where `dt` is a separate column that *you* must fill in on every insert and that every query must filter on. Here `days(tpep_pickup_datetime)` is a **transform** of an existing column: Iceberg computes it on write and applies it on read. Other transforms: `months()`, `years()`, `hours()`, `bucket(16, id)`, `truncate(10, name)`.

In [ ]:
columns = ",\n  ".join(f"{f.name} {f.dataType.simpleString()}" for f in jan.schema.fields)
ddl = f'''
CREATE TABLE {TABLE} (
  {columns}
)
USING iceberg
PARTITIONED BY (days(tpep_pickup_datetime))
'''
print(ddl)
spark.sql(ddl)

In [ ]:
s3_tree(PREFIX)

**What to notice:** the table exists, but storage holds **one file**: `metadata/00000-….metadata.json`. No data files yet. Let's open it.

In [ ]:
def current_metadata():
    # The newest metadata.json is the current one (they're numbered 00000, 00001, ...).
    key = [k for k, _ in s3_ls(PREFIX + "metadata/") if k.endswith(".metadata.json")][-1]
    return key, s3_json(key)

key, meta = current_metadata()
print(key)
print("format-version:     ", meta["format-version"])
print("current-snapshot-id:", meta.get("current-snapshot-id"))
print("snapshots:          ", meta.get("snapshots", []))
print("partition spec:     ", meta["partition-specs"][0]["fields"])
print("first 3 columns:    ", [(f["id"], f["name"], f["type"]) for f in meta["schemas"][0]["fields"][:3]])

**What to notice**
- `current-snapshot-id` is `-1` (or missing) and `snapshots` is empty: a brand-new table has **no versions** yet.
- The partition spec says: `transform: day`, `source-id: 2` — "partition by the day of column **2**". Iceberg refers to columns by **ID**, not by name. Remember that for section 7.
- `format-version: 2` is the default. It supports row-level deletes. (Version 3 adds deletion vectors and new types; you opt in per table.)

> **Coming from Hive/Impala:** metadata.json holds what the metastore's table and partition rows hold in Hive (schema, partition layout) — but as a file in storage next to the data, plus the version history Hive never had.

## 2. Insert January → the first snapshot

In [ ]:
jan.writeTo(TABLE).append()
spark.table(TABLE).count()      # answered from metadata: manifests store each file's row count

# Same as SQL:  INSERT INTO learn.yellow_trips SELECT * FROM jan   (after jan.createOrReplaceTempView("jan"))

In [ ]:
s3_tree(PREFIX)

**What to notice**
- `data/` now has one folder per day — `tpep_pickup_datetime_day=2024-01-15/` — including the odd 2002 / 2009 / Dec-2023 / Feb-2024 days from notebook 01.
- `metadata/` gained: a new `00001-….metadata.json`, one `snap-….avro` (the **manifest list**), and `…-m0.avro` (a **manifest**).

Iceberg exposes its metadata as **tables you can query** — `<table>.snapshots`, `.files`, `.manifests`, `.history`, `.partitions`.

> **Coming from Hive/Impala:** the `data/` folders *look* like Hive partitions (`name=value/`), but Iceberg never reads folder names — it finds files through the manifests. The metadata tables replace Impala's `SHOW FILES`, `SHOW PARTITIONS` and `SHOW TABLE STATS`, except you can query, filter and join them with SQL.

In [ ]:
spark.sql(f'''
  SELECT snapshot_id, operation,
         summary['added-data-files'] AS added_files,
         summary['added-records']    AS added_rows,
         summary['total-records']    AS total_rows
  FROM {TABLE}.snapshots
''').show(truncate=False)

In [ ]:
# One row per data file. Iceberg recorded row counts and min/max for EVERY column —
# even though the source file from TLC had no statistics at all (notebook 01, section 1).
files = spark.sql(f'''
  SELECT partition.tpep_pickup_datetime_day AS day,
         record_count,
         round(file_size_in_bytes / 1024 / 1024, 2) AS size_mb,
         readable_metrics.fare_amount.lower_bound AS fare_min,
         readable_metrics.fare_amount.upper_bound AS fare_max
  FROM {TABLE}.files
  ORDER BY day
''')
print(files.count(), "data files")
files.show(8)

**What to notice:** the manifests hold per-file stats (`fare_min`, `fare_max`, row counts). The engine can skip whole files **before opening them** — no need for stats inside each Parquet footer.

## 3. Skipping files: hidden partitioning and min/max stats

`scan_report(df)` runs a query and reports how many data files Spark **read** vs **skipped**. These are the same numbers the Spark UI shows (http://localhost:4040 → SQL tab → the `BatchScan` box).

In [ ]:
trips = spark.table(TABLE)
queries = {
    "whole table":        trips,
    "one day (pickup)":   trips.filter("tpep_pickup_datetime >= '2024-01-15' AND tpep_pickup_datetime < '2024-01-16'"),
    "fare_amount > 1000": trips.filter("fare_amount > 1000"),
}
report = pd.DataFrame({name: scan_report(q) for name, q in queries.items()}).T
report[["rows", "data_files_read", "data_files_skipped", "manifests_read", "MB_read"]].convert_dtypes()

**What to notice**
- **One day:** you filtered on the real `tpep_pickup_datetime` column, and Spark read **1 file** and skipped the rest. In notebook 01 the same filter read **every** folder unless you knew about `pickup_date`. Iceberg turns the timestamp filter into a day filter for you, using the partition spec — that's hidden partitioning.
- **fare_amount > 1000:** `fare_amount` is not a partition column, yet most files were skipped. Iceberg compared the filter with each file's `fare_max` in the manifests (section 2) and never opened files that couldn't match.
- `manifests_read`: Iceberg first reads the small manifests to decide which files to open. Fewer, well-organized files make this planning fast.

> **Coming from Hive/Impala:** this is Impala's partition pruning and Parquet row-group skipping combined — but decided **per file, before opening any file**, from metadata Iceberg collected automatically on write. No `COMPUTE STATS` needed.

## 4. Append February → a second snapshot

In [ ]:
feb.writeTo(TABLE).append()

spark.sql(f'''
  SELECT h.made_current_at, s.snapshot_id, s.operation,
         s.summary['added-records'] AS added_rows, s.summary['total-records'] AS total_rows
  FROM {TABLE}.history h JOIN {TABLE}.snapshots s ON h.snapshot_id = s.snapshot_id
  ORDER BY h.made_current_at
''').show(truncate=False)

key, meta = current_metadata()
print(key.split("/")[-1], "→", len(meta["snapshots"]), "snapshots listed in the current metadata.json")

**What to notice:** each write made **one new snapshot** and **one new metadata.json** (`00002-…`). The January files were not touched or copied — the new snapshot's manifest list simply points to the old manifest *plus* a new one for February.

## 5. Time travel

Every snapshot is still queryable. `VERSION AS OF <snapshot_id>` reads the table exactly as it was.

In [ ]:
snaps = [r.snapshot_id for r in spark.sql(f"SELECT snapshot_id FROM {TABLE}.snapshots ORDER BY committed_at").collect()]
first_snapshot = snaps[0]

print("now:                  ", spark.table(TABLE).count())
print("as of first snapshot: ", spark.sql(f"SELECT count(*) FROM {TABLE} VERSION AS OF {first_snapshot}").first()[0])

**What to notice:** the old version is still there because the old metadata and data files were never deleted. Time travel is just "start from an older metadata.json / manifest list". (Old snapshots are cleaned up on purpose with `expire_snapshots` — a later notebook.)

You can also travel by time: `SELECT ... FROM table TIMESTAMP AS OF '2026-10-27 10:00:00'`.

> **Coming from Hive/Impala:** Hive has no equivalent. The closest is an HDFS snapshot of the whole directory, which an admin has to set up and restore. Here every write is a version, and reading an old one is just SQL.

## 6. DELETE — what plain Parquet couldn't do

In [ ]:
print("negative fares before:", spark.table(TABLE).filter("fare_amount < 0").count())
spark.sql(f"DELETE FROM {TABLE} WHERE fare_amount < 0")
print("negative fares after: ", spark.table(TABLE).filter("fare_amount < 0").count())

spark.sql(f'''
  SELECT snapshot_id, operation,
         summary['deleted-data-files'] AS files_removed,
         summary['deleted-records']    AS rows_in_removed_files,
         summary['added-data-files']   AS files_written,
         summary['added-records']      AS rows_rewritten,
         summary['total-records']      AS total_rows
  FROM {TABLE}.snapshots ORDER BY committed_at
''').show(truncate=False)

**What to notice**
- `DELETE` worked and was **atomic**: readers see either all negative fares or none, never a half-finished state. The snapshot's operation is `overwrite` because files were replaced.
- Look at the cost. To remove ~78 thousand rows, Iceberg **rewrote almost every file** — millions of rows (`rows_rewritten`). Parquet files are immutable, so each file containing even one negative fare was copied without those rows. This strategy is **copy-on-write**: expensive writes, fastest reads. Negative fares are spread across every day, so almost every file was hit.
- The alternative, **merge-on-read**, writes small "delete files" instead and applies them at read time: cheap writes, slightly slower reads. A later notebook compares them.
- The old files are still in storage; only the new snapshot stops pointing to them.

> **Coming from Hive/Impala:** copy-on-write is what you'd do by hand with `INSERT OVERWRITE` of each affected partition — Iceberg does it for you, atomically, file by file. Merge-on-read is the idea behind Hive ACID's delta files (and their compaction).

## 7. Undo: rollback to an earlier snapshot

`rollback_to_snapshot` just moves the "current snapshot" pointer back. No data is copied.

In [ ]:
before_delete = snaps[-1]        # the snapshot right after February was appended
spark.sql(f"CALL lake.system.rollback_to_snapshot('{TABLE}', {before_delete})").show()
print("negative fares after rollback:", spark.table(TABLE).filter("fare_amount < 0").count())

**What to notice:** the deleted rows are back instantly — rollback only changed metadata. This is how you recover from a bad pipeline run.

> **Coming from Hive/Impala:** in Hadoop, undoing a bad `INSERT OVERWRITE` means restoring from a backup or HDFS snapshot — if one exists. Here it's one SQL call.

Redo the delete so the table is clean for the next section:

In [ ]:
spark.sql(f"DELETE FROM {TABLE} WHERE fare_amount < 0")
spark.table(TABLE).filter("fare_amount < 0").count()

## 8. Rename a column — safely

In notebook 01, renaming `Airport_fee` split the data into two half-empty columns. In Iceberg:

In [ ]:
spark.sql(f"ALTER TABLE {TABLE} RENAME COLUMN Airport_fee TO airport_surcharge")

(spark.table(TABLE)
      .groupBy(F.month("tpep_pickup_datetime").alias("month"))
      .agg(F.count("airport_surcharge").alias("airport_surcharge_values"))
      .filter("month IN (1, 2)").orderBy("month").show())

key, meta = current_metadata()
latest_schema = meta["schemas"][-1]
print([(f["id"], f["name"]) for f in latest_schema["fields"] if "airport" in f["name"].lower()])

**What to notice**
- Both months still have their values under the **new** name. No data file was rewritten — only a new schema was added to metadata.json.
- The column kept its **ID**. Old Parquet files still say `Airport_fee` inside, but Iceberg maps them by ID, so the name in the file doesn't matter.

> **Coming from Hive/Impala:** in Hive, `ALTER TABLE ... CHANGE COLUMN` only edits the metastore, so old Parquet files (matched by name) return NULL for the renamed column. Iceberg writes a **field ID** into every Parquet file it creates and matches columns by that ID, so renames, drops and reorders are always safe.

## Summary

| You did | What Iceberg wrote | Cost |
|---|---|---|
| CREATE TABLE | 1 metadata.json | metadata only |
| append January | data files + manifest + manifest list + metadata.json | writes new data |
| append February | new files for Feb only; old files reused | writes new data |
| time travel | nothing — reads an old snapshot | free |
| DELETE | rewrote affected files (copy-on-write) + new snapshot | rewrites affected files |
| rollback | moved the current-snapshot pointer | metadata only |
| RENAME COLUMN | new schema in metadata.json | metadata only |

## Exercises
1. In the RustFS console, open `warehouse/learn/yellow_trips/metadata/`. Count the `metadata.json` files. Why is there one per commit, not one per table?
2. Query `learn.yellow_trips.partitions`. Which day has the most rows? How many partitions belong to dates outside Jan–Feb 2024?
3. Use `TIMESTAMP AS OF` (take a `made_current_at` value from the history table) to count rows *before* the first DELETE.
4. Run `ALTER TABLE learn.yellow_trips ADD COLUMN tip_pct double`. What's in `tip_pct` for existing rows? Did it create a new snapshot (check `.snapshots`) or a new metadata.json (check `s3_tree`)? What does that tell you about the cost of adding a column?
5. Run `scan_report` for `payment_type = 3`. Why can't Iceberg skip many files for this filter, even though few rows match? (Hint: look at `lower_bound`/`upper_bound` for `payment_type` in `.files`.)

Next: building a proper **bronze → silver → gold** pipeline with these tables.